In [ ]:
!pip install rdflib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 565.1/565.1 kB 15.3 MB/s eta 0:00:00


In [ ]:
import numpy as np
import rdflib
from rdflib import Graph, Literal, URIRef
from rdflib.namespace import RDF, XSD
from datetime import datetime, timedelta
import random
import hashlib

# Define namespaces
ex = rdflib.Namespace("http://example.org/")
health = rdflib.Namespace("http://example.org/health/")
finance = rdflib.Namespace("http://example.org/finance/")
general = rdflib.Namespace("http://example.org/general/")
time = rdflib.Namespace("http://example.org/time/")

# Create an empty graph
g = Graph()

# Bind namespaces to prefixes for better readability in the output
g.bind("ex", ex)
g.bind("health", health)
g.bind("finance", finance)
g.bind("general", general)
g.bind("time", time)

# Set the start date for our simulation
start_date = datetime(2024, 10, 20, 10, 0, 0) # October 20, 2024, 10:00 AM IST
num_months = 60
current_date = start_date

# Lists of entities and relations for each domain
persons = [ex["JohnDoe"], ex["JaneSmith"], ex["Robert"], ex["Alice"]]
diseases = [health["Flu"], health["COVID19"], health["Allergy"]]
symptoms = [health["Fever"], health["Cough"], health["SoreThroat"], health["Fatigue"], health["RunnyNose"]]
medications = [health["Paracetamol"], health["Ibuprofen"], health["Antihistamine"]]
doctors = [health["DrAlice"], health["DrBob"]]
hospitals = [health["CityHospital"], health["GeneralClinic"]]

accounts = [finance["JohnChecking"], finance["JaneSavings"], finance["RobertCredit"]]
transactions = [finance["T1"], finance["T2"], finance["T3"], finance["T4"], finance["T5"]]
financial_products = [finance["StockA"], finance["BondB"], finance["MutualFundC"]]
banks = [finance["CityBank"], finance["NationalCreditUnion"]]

locations = [general["Home"], general["Office"], general["Gym"], general["Restaurant"]]
activities = [general["Meeting"], general["Workout"], general["Lunch"], general["Travel"]]
items = [general["BookA"], general["Laptop"], general["Phone"]]

relations_health = [health["hasSymptom"], health["diagnosedWith"], health["prescribed"], health["visitedDoctor"], health["atHospital"]]
relations_finance = [finance["hasAccount"], finance["madeTransaction"], finance["investedIn"], finance["accountAt"], finance["transactionAmount"]]
relations_general = [general["locatedAt"], general["participatedIn"], general["ownsItem"]]

# Function to add a temporal triple
def add_temporal_triple(subject, predicate, object_, timestamp):
    g.add((subject, predicate, object_))
    g.add((subject, time["timestamp"], Literal(timestamp.isoformat(), datatype=XSD.dateTime)))

    #stmt_id = URIRef(f"{str(subject.split('/')[-1])}_{str(predicate.split('/')[-1])}_{str(object_.split('/')[-1])}")
    #g.add((stmt_id, time["timestamp"], Literal(timestamp.isoformat(), datatype=XSD.dateTime)))

    # Create a unique ID for the statement
    # raw = f"{str(subject)}_{str(predicate)}_{str(object_)}"
    # stmt_id = URIRef(f"http://example.org/event/{hashlib.md5(raw.encode()).hexdigest()}")


    # # Attach timestamp to this statement identifier
    # g.add((stmt_id, time["timestamp"], Literal(timestamp.isoformat(), datatype=XSD.dateTime)))


# Simulate events over 6 months
for month in range(num_months):
    days_in_month = (current_date.replace(month=current_date.month % 12 + 1, day=1) - timedelta(days=1)).day
    for day in range(days_in_month):
        for hour in range(random.randint(1, 5)): # Simulate a few events per day
            minute = random.randint(0, 59)
            second = random.randint(0, 59)
            event_time = current_date.replace(day=day + 1, hour=random.randint(8, 18), minute=minute, second=second)

            # Simulate Health events
            if random.random() < 0.15:
                person = random.choice(persons)
                if random.random() < 0.4:
                    disease_or_symptom = random.choice(diseases + symptoms)
                    relation = random.choice([health["hasSymptom"], health["diagnosedWith"]])
                    add_temporal_triple(person, relation, disease_or_symptom, event_time)
                elif random.random() < 0.3:
                    person = random.choice(persons)
                    med = random.choice(medications)
                    doctor = random.choice(doctors)
                    add_temporal_triple(doctor, health["prescribed"], med, event_time)
                    add_temporal_triple(person, health["prescribed"], med, event_time)
                    add_temporal_triple(person, health["visitedDoctor"], doctor, event_time)
                elif random.random() < 0.2:
                    person = random.choice(persons)
                    hospital = random.choice(hospitals)
                    add_temporal_triple(person, health["atHospital"], hospital, event_time)

            # Simulate Finance events
            if random.random() < 0.2:
                person = random.choice(persons)
                account = random.choice(accounts)
                relation_fin = random.choice(relations_finance[:-1]) # Exclude transactionAmount initially
                add_temporal_triple(person, finance["hasAccount"], account, event_time)
                if random.random() < 0.3:
                    transaction = random.choice(transactions)
                    add_temporal_triple(account, finance["madeTransaction"], transaction, event_time)
                    amount = round(random.uniform(10, 1000), 2)
                    add_temporal_triple(transaction, finance["transactionAmount"], Literal(amount, datatype=XSD.float), event_time)
                elif random.random() < 0.1:
                    product = random.choice(financial_products)
                    add_temporal_triple(person, finance["investedIn"], product, event_time)
                elif random.random() < 0.2:
                    bank = random.choice(banks)
                    add_temporal_triple(account, finance["accountAt"], bank, event_time)

            # Simulate General events
            if random.random() < 0.3:
                person = random.choice(persons)
                location = random.choice(locations)
                add_temporal_triple(person, general["locatedAt"], location, event_time)
            elif random.random() < 0.25:
                person = random.choice(persons)
                activity = random.choice(activities)
                add_temporal_triple(person, general["participatedIn"], activity, event_time)
            elif random.random() < 0.1:
                person = random.choice(persons)
                item = random.choice(items)
                add_temporal_triple(person, general["ownsItem"], item, event_time)

    current_date += timedelta(days=30) # Approximate month increment

small_g = g

In [ ]:
# Re-import necessary libraries due to kernel reset
import numpy as np
import rdflib
from rdflib import Graph, Literal, URIRef
from rdflib.namespace import RDF, XSD
from datetime import datetime, timedelta
import random

# Define the function again after reset
def generate_rich_tkg(events_per_day=10, temporal_span_months=12, seed=42):
    random.seed(seed)
    np.random.seed(seed)

    # Define namespaces
    ex = rdflib.Namespace("http://example.org/")
    health = rdflib.Namespace("http://example.org/health/")
    finance = rdflib.Namespace("http://example.org/finance/")
    general = rdflib.Namespace("http://example.org/general/")
    time = rdflib.Namespace("http://example.org/time/")

    # Create an empty graph
    g = Graph()
    g.bind("ex", ex)
    g.bind("health", health)
    g.bind("finance", finance)
    g.bind("general", general)
    g.bind("time", time)

    # Static background entities
    persons = [ex[name] for name in ["JohnDoe", "JaneSmith", "Robert", "Alice", "Emily", "Michael"]]
    static_types = [
        (RDF.type, ex.Person),
        (ex.gender, Literal(random.choice(["Male", "Female"]))),
        (ex.age, Literal(random.randint(20, 60))),
    ]

    # Add static background triples
    for person in persons:
        for pred, obj in static_types:
            g.add((person, pred, obj))

    # Entity pools
    diseases = [health[d] for d in ["Flu", "COVID19", "Allergy"]]
    symptoms = [health[s] for s in ["Fever", "Cough", "SoreThroat", "Fatigue", "RunnyNose"]]
    medications = [health[m] for m in ["Paracetamol", "Ibuprofen", "Antihistamine"]]
    doctors = [health[d] for d in ["DrAlice", "DrBob"]]
    hospitals = [health[h] for h in ["CityHospital", "GeneralClinic"]]

    accounts = [finance[a] for a in ["JohnChecking", "JaneSavings", "RobertCredit", "EmilyLoan"]]
    transactions = [finance[f"T{i}"] for i in range(1, 11)]
    financial_products = [finance[p] for p in ["StockA", "BondB", "MutualFundC"]]
    banks = [finance[b] for b in ["CityBank", "NationalCreditUnion"]]

    locations = [general[l] for l in ["Home", "Office", "Gym", "Restaurant", "Park"]]
    activities = [general[a] for a in ["Meeting", "Workout", "Lunch", "Travel", "Coffee"]]
    items = [general[i] for i in ["BookA", "Laptop", "Phone", "Bike"]]

    # Relations
    health_rel = [health.hasSymptom, health.diagnosedWith, health.prescribed, health.visitedDoctor, health.atHospital]
    finance_rel = [finance.hasAccount, finance.madeTransaction, finance.investedIn, finance.accountAt, finance.transactionAmount]
    general_rel = [general.locatedAt, general.participatedIn, general.ownsItem, general.metWith]

    # Helper: Add temporal triple
    def add_temporal_triple(subject, predicate, object_, timestamp):
        g.add((subject, predicate, object_))
        g.add((subject, time.timestamp, Literal(timestamp.isoformat(), datatype=XSD.dateTime)))

    # Simulate events
    current_date = datetime(2024, 1, 1, 9, 0, 0)
    for month in range(temporal_span_months):
        for day in range(28):  # simplify to 28 days/month
            for _ in range(events_per_day):
                event_time = current_date + timedelta(days=day, hours=random.randint(8, 18), minutes=random.randint(0, 59))
                person = random.choice(persons)

                # Health
                if random.random() < 0.4:
                    disease_or_symptom = random.choice(diseases + symptoms)
                    rel = random.choice([health.hasSymptom, health.diagnosedWith])
                    add_temporal_triple(person, rel, disease_or_symptom, event_time)

                    if random.random() < 0.5:
                        doctor = random.choice(doctors)
                        med = random.choice(medications)
                        add_temporal_triple(doctor, health.prescribed, med, event_time)
                        add_temporal_triple(person, health.prescribed, med, event_time)
                        add_temporal_triple(person, health.visitedDoctor, doctor, event_time)
                        add_temporal_triple(person, health.atHospital, random.choice(hospitals), event_time)

                # Finance
                if random.random() < 0.4:
                    account = random.choice(accounts)
                    transaction = random.choice(transactions)
                    add_temporal_triple(person, finance.hasAccount, account, event_time)
                    add_temporal_triple(account, finance.madeTransaction, transaction, event_time)
                    add_temporal_triple(transaction, finance.transactionAmount,
                                        Literal(round(random.uniform(20, 5000), 2), datatype=XSD.float), event_time)

                    if random.random() < 0.3:
                        add_temporal_triple(person, finance.investedIn, random.choice(financial_products), event_time)
                    if random.random() < 0.3:
                        add_temporal_triple(account, finance.accountAt, random.choice(banks), event_time)

                # General Life Events
                if random.random() < 0.5:
                    location = random.choice(locations)
                    activity = random.choice(activities)
                    add_temporal_triple(person, general.locatedAt, location, event_time)
                    add_temporal_triple(person, general.participatedIn, activity, event_time)

                    if random.random() < 0.3:
                        other = random.choice([p for p in persons if p != person])
                        add_temporal_triple(person, general.metWith, other, event_time)
                    if random.random() < 0.2:
                        item = random.choice(items)
                        add_temporal_triple(person, general.ownsItem, item, event_time)

        current_date += timedelta(days=30)

    return g

# Generate a richly populated TKG
rich_g = generate_rich_tkg(events_per_day=45, temporal_span_months=12)
len(rich_g)  # Show number of triples generated


33510

In [ ]:
g.serialize(destination="simulated_tkg.ttl", format="turtle")

<Graph identifier=N4848bc5d2aa94918b9904183abcb9288 (<class 'rdflib.graph.Graph'>)>

In [ ]:
len(g)

5527

Subgraph extraction

In [ ]:
!pip install torch pykeen


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.5/85.5 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 121.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 90.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 13.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 71.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 718.4/718.4 kB 49.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
from rdflib import Graph, URIRef, Literal
from pykeen.pipeline import pipeline
import torch
from collections import defaultdict
import random
import hashlib

from datetime import datetime


# # Load the TKG
# g = Graph()
# g.parse("simulated_tkg.ttl", format="ttl")

def whole_g_snapshot(graph):
    snapshot = Graph()
    for triple in graph:
        snapshot.add(triple)
    return snapshot

# # Extract snapshot at time t (you can also define a time window)
# def extract_snapshot(graph, time_literal):
#     snapshot = Graph()
#     for s, p, o in graph:
#         if "timestamp" in str(p) and str(o) == time_literal:
#             snapshot.add((s, p, o))
#             for triple in graph.triples((s, None, None)):
#                 snapshot.add(triple)
#     return snapshot

# def extract_snapshot(graph, time_literal):
#     snapshot = Graph()

#     valid_hashes = set()
#     for stmt_id, _, ts in graph.triples((None, time["timestamp"], None)):
#         if str(ts) == time_literal:
#             hash_val = str(stmt_id).split("/")[-1]
#             valid_hashes.add(hash_val)

#     for s, p, o in graph:
#         if not isinstance(o, URIRef):
#             continue
#         raw = f"{str(s)}_{str(p)}_{str(o)}"
#         hash_val = hashlib.md5(raw.encode()).hexdigest()
#         if hash_val in valid_hashes:
#             snapshot.add((s, p, o))

#     return snapshot

def extract_snapshot(graph, time_literal):
    snapshot = Graph()

    valid_hashes = set()
    for stmt_id, _, ts in graph.triples((None, time["timestamp"], None)):
        if str(ts) == time_literal:
            hash_val = str(stmt_id).split("/")[-1]
            valid_hashes.add(hash_val)

    for s, p, o in graph:
        raw = f"{str(s)}_{str(p)}_{str(o)}"
        hash_val = hashlib.md5(raw.encode()).hexdigest()
        if hash_val in valid_hashes:
            snapshot.add((s, p, o))

    return snapshot

def extract_bucketed_snapshot_simple(graph, timestamps, target_index, granularity="day"):
    # Group timestamps into buckets
    def bucket_key(ts_str):
        ts = datetime.fromisoformat(ts_str)
        if granularity == "week":
            return f"{ts.year}-W{ts.isocalendar().week}"
        elif granularity == "month":
            return f"{ts.year}-{ts.month:02d}"
        else:  # default: day
            return ts.date().isoformat()

    grouped = defaultdict(list)
    for ts in timestamps:
        grouped[bucket_key(ts)].append(ts)

    # Get selected bucket
    bucket_keys = sorted(grouped)

    if target_index >= len(bucket_keys):
        raise IndexError("Snapshot index out of range")

    selected_timestamps = set(grouped[bucket_keys[target_index]])

    # Find subjects with matching timestamps
    valid_subjects = set()
    for s, p, o in graph.triples((None, time["timestamp"], None)):
        if str(o) in selected_timestamps:
            valid_subjects.add(s)

    # Collect all triples with matching subjects
    snapshot = Graph()
    for s, p, o in graph:
        if s in valid_subjects and p != time["timestamp"]:
            snapshot.add((s, p, o))

    return snapshot

def extract_bucketed_snapshot(graph, timestamps, target_index, granularity="day"):
    # Group timestamps into buckets
    def bucket_key(ts_str):
        ts = datetime.fromisoformat(ts_str)
        if granularity == "week":
            return f"{ts.year}-W{ts.isocalendar().week}"
        elif granularity == "month":
            return f"{ts.year}-{ts.month:02d}"
        else:  # default: day
            return ts.date().isoformat()

    grouped = defaultdict(list)
    for ts in timestamps:
        grouped[bucket_key(ts)].append(ts)

    # Get target bucket by index
    bucket_keys = sorted(grouped)
    if target_index >= len(bucket_keys):
        raise IndexError("Target index out of range")

    selected_bucket = grouped[bucket_keys[target_index]]

    # Collect triples for all timestamps in the selected bucket
    snapshot = Graph()
    valid_hashes = set()

    for ts in selected_bucket:
        for stmt_id, _, val in graph.triples((None, time["timestamp"], None)):
            if str(val) == ts:
                hash_val = str(stmt_id).split("/")[-1]
                valid_hashes.add(hash_val)

    for s, p, o in graph:
        raw = f"{str(s)}_{str(p)}_{str(o)}"
        hash_val = hashlib.md5(raw.encode()).hexdigest()
        if hash_val in valid_hashes:
            snapshot.add((s, p, o))

    return snapshot

from datetime import datetime
from collections import defaultdict

def group_timestamps_by_bucket(timestamps, granularity="week"):
    def bucket_key(ts_str):
        ts = datetime.fromisoformat(ts_str)
        if granularity == "week":
            return f"{ts.year}-W{ts.isocalendar().week}"
        elif granularity == "month":
            return f"{ts.year}-{ts.month:02d}"
        else:
            return ts.date().isoformat()

    grouped = defaultdict(list)
    for ts in timestamps:
        grouped[bucket_key(ts)].append(ts)

    bucket_keys = sorted(grouped)
    return grouped, bucket_keys


In [ ]:
def rdf_to_triples(graph):
    triples = []
    for s, p, o in graph:
        if isinstance(o, URIRef):
            triples.append((str(s), str(p), str(o)))
    return triples


# Build namespace prefix map
prefix_map = {str(ns): prefix for prefix, ns in g.namespace_manager.namespaces()}

# Global label shortener
def prefixed_label(uri):
    uri_str = str(uri)
    for ns_uri, prefix in prefix_map.items():
        if uri_str.startswith(ns_uri):
            return f"{prefix}:{uri_str[len(ns_uri):]}"
    return uri_str  # fallback to full URI if no match



In [ ]:
#### Candidate note/triple selection

def identify_candidates(triples, top_k=5):
    degree_count = defaultdict(int)
    for s, _, o in triples:
        degree_count[s] += 1
        degree_count[o] += 1
    top_nodes = sorted(degree_count, key=degree_count.get, reverse=True)[:top_k]
    return top_nodes


In [ ]:
# !pip uninstall -y numpy gensim
!pip install numpy==1.23.5 gensim --force-reinstall
!pip install node2vec

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.1/17.1 MB 115.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.7/26.7 MB 89.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.6/38.6 MB 20.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.7/61.7 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.2/83.2 kB 8.5 MB/s eta 0:00:00
  Attempting uninstall: wrapt
    Found existing installation: wrapt 1.17.2
    Uninstalling wrapt-1.17.2:
      Successfully uninstalled wrapt-1.17.2
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: smart-open
    Found existing installation: smart-open 7.1.0
    Uninstalling smart-open-7.1.0:
      Successfully uninstalled smart-open-7.1.0
  Attempting uninstall: scipy
    Found existing installati

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 20.7 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 1.23.5
    Uninstalling numpy-1.23.5:
      Successfully uninstalled numpy-1.23.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.26.4 which is incompatible.


In [ ]:
### subgraph expansion

import networkx as nx
from node2vec import Node2Vec
from rdflib import URIRef

def triples_to_nx_graph(triples):
    G = nx.DiGraph()
    for h, r, t in triples:
        G.add_edge(str(h), str(t), label=str(r))
    return G

def get_embeddings_and_expand_simple(triples, candidate_nodes, max_depth=2, threshold=0.1):
    # Convert RDF triples to a NetworkX graph
    G = triples_to_nx_graph(triples)

    # Generate embeddings using Node2Vec (no training, unsupervised)
    node2vec = Node2Vec(G, dimensions=64, walk_length=10, num_walks=50, workers=1)
    model = node2vec.fit()

    # Normalize embeddings
    def similarity(n1, n2):
        try:
            return model.wv.similarity(str(n1), str(n2))
        except KeyError:
            return 0.0

    subgraphs = []
    for root in candidate_nodes:
        visited = set([root])
        frontier = [root]
        subgraph = []

        for _ in range(max_depth):
            next_frontier = []
            for h, r, t in triples:
                h_str, t_str = str(h), str(t)
                if h_str in frontier:
                    score = similarity(root, t_str)
                    #print ('similarity score: ', score)
                    if score >= threshold:
                        subgraph.append((h, r, t))
                        if t_str not in visited:
                            next_frontier.append(t_str)
                            visited.add(t_str)
            frontier = next_frontier

        if subgraph:
            subgraphs.append(subgraph)

    return subgraphs


In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from node2vec import Node2Vec
import networkx as nx

# Load MiniLM model globally (only once)
semantic_model = SentenceTransformer("all-MiniLM-L6-v2")

def triples_to_nx_graph(triples):
    G = nx.DiGraph()
    for h, r, t in triples:
        G.add_edge(str(h), str(t), label=str(r))
    return G

def get_embeddings_and_expand_with_semantics(triples, candidate_nodes, max_depth=2, threshold=0.5, semantic_threshold=0.6):
    # Step 1: Build networkx graph
    G = triples_to_nx_graph(triples)

    # Step 2: Node2Vec embeddings
    node2vec = Node2Vec(G, dimensions=64, walk_length=10, num_walks=50, workers=1)
    model = node2vec.fit()

    def similarity(n1, n2):
        try:
            return model.wv.similarity(str(n1), str(n2))
        except KeyError:
            return 0.0

    # Step 3: Semantic filtering
    def is_semantically_similar(relation_uri, core_embedding, threshold):
        rel_label = relation_uri.split("/")[-1]
        emb = semantic_model.encode(rel_label)
        sim = cosine_similarity([core_embedding], [emb])[0][0]
        return sim >= threshold

    subgraphs = []

    for root in candidate_nodes:
        visited = set([root])
        frontier = [root]
        subgraph = []

        # Core embedding based on most frequent predicate around root
        root_triples = [triple for triple in triples if str(triple[0]) == root]
        if not root_triples:
            continue
        top_rel = max(root_triples, key=lambda x: sum(x[1] in r for _, r, _ in root_triples))[1]
        top_rel_label = top_rel.split("/")[-1]
        core_embedding = semantic_model.encode(top_rel_label)

        for _ in range(max_depth):
            next_frontier = []
            for h, r, t in triples:
                h_str, t_str = str(h), str(t)
                if h_str in frontier:
                    struct_score = similarity(root, t_str)
                    if struct_score >= threshold:
                        if is_semantically_similar(r, core_embedding, semantic_threshold):
                            subgraph.append((h, r, t))
                            if t_str not in visited:
                                next_frontier.append(t_str)
                                visited.add(t_str)
            frontier = next_frontier

        if subgraph:
            subgraphs.append(subgraph)

    return subgraphs


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.metrics.pairwise import cosine_similarity
from collections import defaultdict
import numpy as np
import networkx as nx
from node2vec import Node2Vec

semantic_model = SentenceTransformer('all-MiniLM-L6-v2')

def triples_to_nx_graph(triples):
    G = nx.DiGraph()
    for h, r, t in triples:
        G.add_edge(str(h), str(t), label=str(r))
    return G

def embed_predicates(triples):
    predicates = [r.split("/")[-1] for _, r, _ in triples]
    embeddings = semantic_model.encode(predicates)
    return embeddings

def get_entropy(embeddings):
    sim_matrix = cosine_similarity(embeddings)
    avg_sim = np.mean(sim_matrix, axis=1)
    diversity = 1 - avg_sim
    return np.mean(diversity)

def expand_and_cluster_aspects(triples, top_k=20, max_depth=10, struct_threshold=0.2, k_clusters=2):
    G = triples_to_nx_graph(triples)

    node2vec = Node2Vec(G, dimensions=64, walk_length=10, num_walks=50, workers=1)
    model = node2vec.fit()

    def similarity(n1, n2):
        try:
            return model.wv.similarity(str(n1), str(n2))
        except KeyError:
            return 0.0

    # --- Step 1: Smart Candidate Selection based on embedding diversity ---
    node_neighbors = defaultdict(list)
    for h, _, t in triples:
        node_neighbors[h].append(t)
        node_neighbors[t].append(h)

    diversity_scores = {}
    for node in node_neighbors:
        local_triples = [triple for triple in triples if str(triple[0]) == node or str(triple[2]) == node]
        if len(local_triples) < 3:
            continue
        emb = embed_predicates(local_triples)
        diversity_scores[node] = get_entropy(emb)

    top_nodes = sorted(diversity_scores, key=diversity_scores.get, reverse=True)[:top_k]

    # --- Step 2: For each top node, extract local subgraph ---
    all_aspect_subgraphs = []

    for root in top_nodes:
        visited = set([root])
        frontier = [root]
        local_triples = []

        for _ in range(max_depth):
            next_frontier = []
            for h, r, t in triples:
                h_str, t_str = str(h), str(t)
                if h_str in frontier:
                    if similarity(root, t_str) >= struct_threshold:
                        local_triples.append((h, r, t))
                        if t_str not in visited:
                            next_frontier.append(t_str)
                            visited.add(t_str)
            frontier = next_frontier

        if not local_triples:
            continue

        # --- Step 3: Embed and cluster predicates into latent aspects ---
        embeddings = embed_predicates(local_triples)

        if len(embeddings) < 2:
            continue  # Not enough for clustering

        # Optional: Auto-tune k using silhouette score
        best_score = -1
        best_k = 2
        for k in range(2, min(6, len(embeddings))):
            kmeans = KMeans(n_clusters=k, random_state=42)
            labels = kmeans.fit_predict(embeddings)
            score = silhouette_score(embeddings, labels)
            if score > best_score:
                best_score = score
                best_k = k

        kmeans = KMeans(n_clusters=best_k, random_state=42)
        labels = kmeans.fit_predict(embeddings)

        # --- Step 4: Split triples into aspect subgraphs ---
        clustered_subgraphs = defaultdict(list)
        for triple, label in zip(local_triples, labels):
            clustered_subgraphs[label].append(triple)

        all_aspect_subgraphs.extend(clustered_subgraphs.values())

    return all_aspect_subgraphs


In [ ]:
from rdflib.namespace import XSD

# Function to extract all unique timestamp literals from the graph
def get_all_timestamps(graph):
    timestamps = set()
    for _, p, o in graph:
        if 'timestamp' in str(p) and isinstance(o, Literal) and o.datatype == XSD.dateTime:
            timestamps.add(str(o))
    return sorted(timestamps)


In [ ]:


# timestamps = get_all_timestamps(g)
# print("Available timestamps:", len(timestamps))
# # for i, ts in enumerate(timestamps):
# #     print(f"{i}. {ts}")

# # # Choose one (manually or programmatically)
# # index = 2  # Example: third snapshot
# # t = timestamps[index]
# # print(f"Using snapshot at: {t}")


# # Example: pick the earliest timestamp
# t = timestamps[150]  # or timestamps[-1] for latest, timestamps[len(timestamps)//2] for middle

# # print(f"Using snapshot at time: {t}")
# print(f"Using snapshot index {index} with granularity='week'")


In [ ]:
# for i in range(len(timestamps)):
#   t = timestamps[i]
#   snapshot = extract_snapshot(g, t)
#   triples = rdf_to_triples(snapshot)
#   if len(triples) > 1:
#     print (i, len(triples))

In [ ]:
g = small_g # small_g or rich_g
# t = timestamps[1777]
timestamps = get_all_timestamps(g)
index = 0

# snapshot = extract_snapshot(g, t)
snapshot = extract_bucketed_snapshot_simple(g, timestamps, index, granularity="week")
# snapshot = whole_g_snapshot(g)

# uncomment for (s,p,o)->time
# snapshot = extract_bucketed_snapshot(g, timestamps, index, granularity="month")  # "week" or "month"

triples = rdf_to_triples(snapshot)
candidates = identify_candidates(triples, top_k=100)
# subgraphs = get_embeddings_and_expand_simple(triples, candidates)
# subgraphs = get_embeddings_and_expand_with_semantics(
#     triples,
#     candidates,
#     max_depth=10,
#     threshold=0.05,              # structural threshold
#     semantic_threshold=0.4      # semantic coherence threshold
# )
subgraphs = expand_and_cluster_aspects(
    triples,
    top_k=10,           # number of candidate roots
    max_depth=15,        # expansion depth
    struct_threshold=0.2,
    k_clusters=2        # optional fixed cluster count (auto-tunes inside)
)


# for idx, sg in enumerate(subgraphs):
#     print(f"\nSubgraph {idx+1}:")
#     for triple in sg:
#         print(triple)
for idx, sg in enumerate(subgraphs):
    print(f"\nSubgraph {idx+1}:")
    for triple in sg:
        print(tuple(prefixed_label(x) for x in triple))


Computing transition probabilities:   0%|          | 0/36 [00:00<?, ?it/s]

Generating walks (CPU: 1): 100%|██████████| 50/50 [00:00<00:00, 3680.18it/s]



Subgraph 1:
('ex:JaneSmith', 'ex:health/prescribed', 'ex:health/Paracetamol')
('ex:JaneSmith', 'ex:health/diagnosedWith', 'ex:health/RunnyNose')

Subgraph 2:
('ex:JaneSmith', 'ex:health/hasSymptom', 'ex:health/RunnyNose')

Subgraph 3:
('ex:Robert', 'ex:general/locatedAt', 'ex:general/Gym')
('ex:Robert', 'ex:finance/investedIn', 'ex:finance/StockA')

Subgraph 4:
('ex:Robert', 'ex:finance/hasAccount', 'ex:finance/RobertCredit')

Subgraph 5:
('ex:JohnDoe', 'ex:finance/investedIn', 'ex:finance/BondB')
('ex:JohnDoe', 'ex:health/prescribed', 'ex:health/Antihistamine')

Subgraph 6:
('ex:JohnDoe', 'ex:general/locatedAt', 'ex:general/Restaurant')
('ex:JohnDoe', 'ex:health/atHospital', 'ex:health/CityHospital')

Subgraph 7:
('ex:Alice', 'ex:health/hasSymptom', 'ex:health/SoreThroat')
('ex:Alice', 'ex:health/hasSymptom', 'ex:health/Fatigue')

Subgraph 8:
('ex:Alice', 'ex:health/diagnosedWith', 'ex:health/Fatigue')
('ex:Alice', 'ex:health/diagnosedWith', 'ex:health/SoreThroat')

Subgraph 9:
('ex:

/usr/local/lib/python3.11/dist-packages/sklearn/base.py:1389: ConvergenceWarning: Number of distinct clusters (3) found smaller than n_clusters (4). Possibly due to duplicate points in X.
  return fit_method(estimator, *args, **kwargs)


In [ ]:
len(triples)

187

TEmporal Evolutions

In [ ]:
# timestamps

In [ ]:
def match_subgraph_in_snapshot(subgraph, snapshot):
    snapshot_triples = set((str(s), str(p), str(o)) for s, p, o in snapshot if isinstance(o, URIRef))
    original = set((str(s), str(p), str(o)) for s, p, o in subgraph)

    overlap = original & snapshot_triples
    added = snapshot_triples - original
    removed = original - snapshot_triples

    return {
        "overlap_count": len(overlap),
        "added_count": len(added),
        "removed_count": len(removed),
        "total_snapshot": len(snapshot_triples),
        "overlap_ratio": len(overlap) / max(1, len(original)),
        "added_triples": added,
        "removed_triples": removed,
    }

def analyze_evolution(subgraph, snapshots):
    stats = []
    for snap in snapshots:
        result = match_subgraph_in_snapshot(subgraph, snap)
        stats.append(result)

    # Compute deltas
    overlaps = [s["overlap_ratio"] for s in stats]
    additions = [s["added_count"] for s in stats]
    removals = [s["removed_count"] for s in stats]

    def trend_score(values):
        return values[-1] - values[0]  # simple delta

    delta_overlap = trend_score(overlaps)
    delta_add = trend_score(additions)
    delta_rem = trend_score(removals)

    # Classification logic (simple heuristics)
    if delta_add > 2 and delta_overlap > 0.5:
        label = "Growing"
    elif delta_rem > delta_add and overlaps[-1] < 0.4:
        label = "Decaying"
    elif max(additions) > 2 and sum(additions) / len(additions) > 1.5:
        label = "Getting stronger"
    elif abs(delta_overlap) < 0.1 and abs(delta_add) < 1:
        label = "Stable"
    else:
        label = "Mixed"

    return {
        "evolution": label,
        "overlaps": overlaps,
        "additions": additions,
        "removals": removals
    }

# def track_subgraph_evolution_over_time(subgraphs, g, timestamps, t0_index):
#     final_outputs = []

#     future_ts = timestamps[t0_index+1:]

#     # snapshots = [extract_snapshot(g, t) for t in future_ts]
#     # snapshots = [extract_bucketed_snapshot_simple(g, timestamps, t, granularity="week") for t in future_ts]
#     snapshots = [
#         extract_bucketed_snapshot_simple(g, timestamps, i, granularity="week")
#         for i in range(t0_index + 1, len(timestamps))
#     ]


#     for idx, subgraph in enumerate(subgraphs):
#         evolution = analyze_evolution(subgraph, snapshots)
#         final_outputs.append({
#             "subgraph_id": idx,
#             "label": evolution["evolution"],
#             "details": evolution,
#             "original_subgraph": subgraph
#         })

#     return final_outputs

def track_subgraph_evolution_over_time(subgraphs, g, timestamps, t0_index, granularity="week"):
    final_outputs = []

    grouped, bucket_keys = group_timestamps_by_bucket(timestamps, granularity)

    future_indices = range(t0_index + 1, len(bucket_keys))
    snapshots = [
        extract_bucketed_snapshot_simple(g, timestamps, i, granularity=granularity)
        for i in future_indices
    ]

    for idx, subgraph in enumerate(subgraphs):
        evolution = analyze_evolution(subgraph, snapshots)
        final_outputs.append({
            "subgraph_id": idx,
            "label": evolution["evolution"],
            "details": evolution,
            "original_subgraph": subgraph
        })

    return final_outputs


In [ ]:
index = 0
evolved_subgraphs = track_subgraph_evolution_over_time(
    subgraphs,
    g,
    timestamps,
    index
)

for s in evolved_subgraphs:
    print(f"Subgraph {s['subgraph_id']}: {s['label']}")


Subgraph 0: Getting stronger
Subgraph 1: Getting stronger
Subgraph 2: Getting stronger
Subgraph 3: Getting stronger
Subgraph 4: Getting stronger
Subgraph 5: Getting stronger
Subgraph 6: Getting stronger
Subgraph 7: Getting stronger
Subgraph 8: Getting stronger


In [ ]:
from pyvis.network import Network
import networkx as nx

def triples_to_nx_graph(triples):
    G = nx.DiGraph()
    for h, r, t in triples:
        G.add_edge(str(h), str(t), label=str(r))
    return G

def visualize_subgraph_evolution_pyvis(full_triples, evolved_subgraphs, filename_prefix="evolution_subgraph"):
    full_G = triples_to_nx_graph(full_triples)

    color_map = {
        "Growing": "green",
        "Decaying": "red",
        "Stable": "blue",
        "Getting stronger": "orange",
        "Mixed": "gray"
    }

    for sub in evolved_subgraphs:
        net = Network(notebook=False, directed=True, height="700px", width="100%")
        subgraph = sub["original_subgraph"]
        label = sub["label"]
        sub_id = sub["subgraph_id"]

        sub_G = triples_to_nx_graph(subgraph)
        sub_color = color_map.get(label, "gray")

        # Add all nodes from full graph in light gray
        for node in full_G.nodes():
            net.add_node(node, label=node.split("/")[-1], color="lightgray")

        # Add full edges in gray
        for u, v, d in full_G.edges(data=True):
            net.add_edge(u, v, label=d['label'].split("/")[-1], color="lightgray")

        # Overlay subgraph nodes and edges in color
        for node in sub_G.nodes():
            net.add_node(node, label=node.split("/")[-1], color=sub_color)
        for u, v, d in sub_G.edges(data=True):
            net.add_edge(u, v, label=d['label'].split("/")[-1], color=sub_color, width=3)

        # Save and show
        filename = f"{filename_prefix}_{sub_id}_{label}.html"
        net.save_graph(filename)
        print(f"Saved: {filename}")


In [ ]:
# Extract snapshot at time T₀ (for full graph context)
snapshot = extract_snapshot(g, timestamps[index])
full_triples = rdf_to_triples(snapshot)

# Visualize each subgraph evolution in PyVis
visualize_subgraph_evolution_pyvis(full_triples, evolved_subgraphs)

Saved: evolution_subgraph_0_Decaying.html
Saved: evolution_subgraph_1_Decaying.html
Saved: evolution_subgraph_2_Decaying.html
Saved: evolution_subgraph_3_Decaying.html
Saved: evolution_subgraph_4_Decaying.html
Saved: evolution_subgraph_5_Decaying.html
Saved: evolution_subgraph_6_Decaying.html
Saved: evolution_subgraph_7_Decaying.html
Saved: evolution_subgraph_8_Decaying.html
Saved: evolution_subgraph_9_Decaying.html
Saved: evolution_subgraph_10_Decaying.html
Saved: evolution_subgraph_11_Decaying.html


In [ ]:
!pip install pyvis networkx


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 28.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 48.1 MB/s eta 0:00:00


In [ ]:
import networkx as nx
from pyvis.network import Network

# Converts RDF-style triples to NetworkX graph
def triples_to_nx_graph(triples):
    G = nx.DiGraph()
    for h, r, t in triples:
        G.add_edge(str(h), str(t), label=str(r))
    return G

# Visualize with PyVis
def visualize_interactive_subgraphs(full_triples, subgraphs):
    G = triples_to_nx_graph(full_triples)
    net = Network(notebook=True, directed=True, height="600px", width="100%")

    # Add full graph in light gray
    for node in G.nodes():
        #net.add_node(node, label=node.split('/')[-1], color="lightgray")
        # Node label
        net.add_node(node, label=prefixed_label(node), color="lightgray")

    for u, v, d in G.edges(data=True):
        # net.add_edge(u, v, label=d['label'], color="lightgray")
        # Edge label
        net.add_edge(u, v, label=prefixed_label(d['label']), color="lightgray")

    # Add subgraphs with distinct colors
    sub_colors = ["red", "blue", "green", "orange", "purple"]
    for i, subgraph in enumerate(subgraphs):
        sgG = triples_to_nx_graph(subgraph)
        color = sub_colors[i % len(sub_colors)]
        for node in sgG.nodes():
            # net.add_node(node, label=node.split('/')[-1], color=color)
            # Subgraph node
            net.add_node(node, label=prefixed_label(node), color=color)

        for u, v, d in sgG.edges(data=True):
            net.add_edge(u, v, label=d['label'], color=color)
            # Subgraph edge
            net.add_edge(u, v, label=prefixed_label(d['label']), color=color)


    #net.show_buttons(filter_=['physics'])  # Optional: interactive controls
    #net.show("tkg_subgraphs.html")  # Saves and opens in browser
    net.save_graph("tkg_subgraphs.html")

# Example usage
visualize_interactive_subgraphs(triples, subgraphs)
